In [1]:
from Bio import SeqIO
import choppy as cp
import primer3
from collections import defaultdict
import re

In [ ]:
all_sequences = list(SeqIO.parse("data/20240414-forSveta.fa", "fasta"))

# seq_names = ["Maizel_COS-AT1G27430-GYF2", "Maizel_COS-SETH5", 
#              "Pereira_COS-SynDNA-f1", "Pereira_COS-SynDNA-f2",
#              "PV252688r_p6utr", "PQ537341r_p6utr,"
#              "PQ488556r_p6utr", "PX021458r_p6utr",
#              "MZ289137_rep", "OR500095r_naive"]

# sequences = [seq for seq in all_sequences if seq.id in seq_names]

sequences = all_sequences

for seq in sequences:
    seq.seq = seq.seq.upper()
    
# I just haven't decided whether I want a list or a dict
sequences_by_id = {seq.id: seq for seq in sequences}

In [ ]:

CONFIG = {
    'kmer_size': 15,
    'max_frag_length': 1000,
    'min_frag_length': 200,
    'min_overlap': 50,
    'max_overlap': 100,
    # The model will end the segment after it reaches this length
    # It is also penalized for not reaching it, though it is possible to end prematurely
    'opt_segment_length': 5000, 
    'min_segment_length': 1000,
    # Space for TT1
    'segment_offset_left': 78,
    'segment_offset_right': 108,
    # Space for TT2
    'seq_offset_left': 106,
    'seq_offset_right': 105,
    # Optimal primer length is 20 (hardcoded below), but this range is generally allowed
    'min_primer_length': 17,
    'max_primer_length': 30,
    # Use to search for mispriming, 
    # misprime Tm is calculated only for matching kmer at 3' end of primer
    'primer_3prime_anchor': 6,
    'max_misprime_tm': 47.0,
    # If overlaps don't differ much, that is the distance between them
    # If there is considerable difference in neighbourhood, this parameter is ignored
    'min_step': 10,
    # Standard primer3 parameters
    'min_gc': 0.3,
    'max_gc': 0.7,
    'min_tm': 57.0,
    'max_tm': 62.0,
    'max_hairpin_tm': 24.0,
    'max_homodimer_tm': 45.0,
    'max_3_self_tm': 35.0,
    'poly_x_pattern': re.compile(r'(A{5,}|T{5,}|G{5,}|C{5,})'),
    # Due to technical reasons, just having a regexp for CGclamp is not enough
    'clamp_length': 3,
    'clamp_pattern': re.compile(r'[GC][AT][GC]|[AT][GC][GC]'),
    # This is the pattern for primers' 5' end. Currenntly allows anything.
    'five_prime_clamp_pattern': re.compile(r'^[ATGC]')
}

In [ ]:
bg_trie = cp.load_trie("../data/S_cerevisiae-R64-GCA_000146045_cat_15.marisa")
seq_tries = {}
for seq in sequences:
    trie = cp.create_kmer_trie(seq, CONFIG['kmer_size'], bg=False)
    seq_tries[seq.id] = trie 

bg_regions = {}
cur_seq_regions = {}
for seq in sequences:
    bg_regions[seq.id] = cp.find_non_homologous_regions(seq, bg_trie, [], 
                                                        CONFIG['kmer_size'], 
                                                        threshold=CONFIG['min_overlap'])
    cur_seq_regions[seq.id] = cp.find_non_homologous_regions(seq, seq_tries[seq.id], bg_trie, 
                                                             CONFIG['kmer_size'], 
                                                             threshold=CONFIG['min_overlap'])

## Functions for primer search

In [ ]:
def reverse_complement(seq):
    return seq[::-1].translate(str.maketrans('ATGC', 'TACG'))

# helper search utilities (assume lists are sorted ascending)
def next_val(a, val, default=None):
    return next((x for x in a if x > val), default)

def prev_val(a, val, default=None):
    return next((x for x in reversed(a) if x < val), default)

def compute_homfree_ranges(seq_str, kmer_size):
    """Return a list of (start,end) homfree ranges for each base in seq_str.

    A position i is annotated with the nearest previous/next k-mer collision
    adjusted to k-mer coordinates, matching the original inline logic.
    """
    kmers = defaultdict(list)
    for i in range(len(seq_str) - kmer_size + 1):
        kmer = seq_str[i:i+kmer_size]
        kmers[kmer].append(i)
        kmers[reverse_complement(kmer)].append(i)

    kmers = {k: v for k, v in kmers.items() if len(v) > 1}

    homfree_ranges = [(0, len(seq_str))] * len(seq_str)

    for i in range(len(seq_str)):
        start, end = homfree_ranges[i]
        if i < len(seq_str) - kmer_size + 1:
            kmer = seq_str[i:i+kmer_size]
            if kmer in kmers:
                n_val = next_val(kmers[kmer], i)
                if n_val is not None:
                    end = min(end, n_val + kmer_size - 1)
        if i >= kmer_size - 1:
            kmer = seq_str[i-kmer_size+1:i+1]
            if kmer in kmers:
                p_val = prev_val(kmers[kmer], i - kmer_size + 1)
                if p_val is not None:
                    start = max(start, p_val + 1)
        homfree_ranges[i] = (start, end)

    return homfree_ranges

def build_anchor_kmers(sequences, anchor_len, reverse_position = True):
    """Builds the 3' k-mer hash map for fast off-target screening."""
    anchor_kmers = defaultdict(list)
    for seq in sequences:
        seq_str = str(seq.seq).upper()
        for pos in range(len(seq_str) - anchor_len + 1):
            kmer_fwd = seq_str[pos:pos + anchor_len]
            kmer_rev = reverse_complement(kmer_fwd)
            
            anchor_kmers[kmer_fwd].append((seq.id, pos + anchor_len - 1, "forward"))
            if reverse_position:
                anchor_kmers[kmer_rev].append((seq.id, len(seq_str) - pos - anchor_len, "reverse"))
            else:
                anchor_kmers[kmer_rev].append((seq.id, pos + anchor_len - 1, "reverse"))
            
    return anchor_kmers

def check_misprime(primer_cand, native_3_prime_pos, side, seq_id, anchor_kmers, sequences_by_id, cfg):
    """Validates the primer against the k-mer map to ensure no high-Tm off-targets."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        # Allow binding to the intended on-target site
        if anchor_side == side and anchor_seq_id == seq_id and anchor_pos == native_3_prime_pos:
            continue
            
        anchor_seq = str(sequences_by_id[anchor_seq_id].seq).upper()
        
        # Extract the off-target sequence depending on strand orientation
        if anchor_side == "forward":
            pos_misprime = anchor_seq[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
        else:
            pos_misprime = reverse_complement(anchor_seq[anchor_pos:min(len(anchor_seq), anchor_pos + cfg['max_primer_length'])])
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True
            
    return False

def check_local_misprime(primer_cand, seq_str, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
    """Checks for potential mispriming within the same potential fragment."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        if anchor_seq_id != seq_id:
            continue
        if anchor_side != side:
            continue
        if anchor_pos == native_3_prime_pos:
            continue
        if anchor_pos - native_3_prime_pos > cfg['max_frag_length'] or native_3_prime_pos > anchor_pos:
            continue

        pos_misprime = seq_str[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True            
    return False

def find_primers_in_regions(seq_record, regions, side, anchor_kmers, sequences_by_id, cfg):
    """Finds primer candidates for a given sequence, regions, and orientation."""
    primer_candidates = []
    seq_id = seq_record.id
    original_seq = str(seq_record.seq).upper()
    
    if side == "forward":
        search_seq = original_seq
        search_regions = regions
    elif side == "reverse":
        search_seq = reverse_complement(original_seq)
        seq_len = len(original_seq)
        search_regions = [(seq_len - r[1], seq_len - r[0]) for r in regions]
    else:
        raise ValueError("Side must be 'forward' or 'reverse'")

    for region in search_regions:
        clamp_matches = cfg['clamp_pattern'].finditer(search_seq, region[0], region[1])
        
        for m in clamp_matches:
            range_start = max(m.start() - (cfg['max_primer_length'] - cfg['clamp_length']), region[0])
            range_end = m.start() - cfg['min_primer_length'] + cfg['clamp_length'] + 1
            
            # Extend 5' -> 3'
            for pr_start in range(range_end - 1, range_start - 1, -1):
                primer_cand = search_seq[pr_start:m.end()]
                if cfg['poly_x_pattern'].search(primer_cand): break
                if not cfg['five_prime_clamp_pattern'].search(primer_cand): continue
                
                gc_content = (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand)
                if gc_content < cfg['min_gc'] or gc_content > cfg['max_gc']: continue
                
                tm = primer3.calc_tm(primer_cand)
                if tm > cfg['max_tm']: break
                if tm < cfg['min_tm']: continue
                if primer3.calc_hairpin_tm(primer_cand) > cfg['max_hairpin_tm']: break
                if primer3.calc_homodimer_tm(primer_cand) > cfg['max_homodimer_tm']: break
                if primer3.calc_end_stability(primer_cand, primer_cand).tm > cfg['max_3_self_tm']: break
                
                if side == "forward":
                    native_3_prime_pos = m.end() - 1
                    pos_tuple = (pr_start, m.end())
                else:
                    native_3_prime_pos = len(original_seq) - m.end()
                    pos_tuple = (len(original_seq) - m.end(), len(original_seq) - pr_start)
                
                if check_local_misprime(primer_cand, search_seq, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
                    break
                    
                primer_candidates.append({
                    'seq': primer_cand,
                    'gc_content': gc_content,
                    'tm': tm,
                    'pos': pos_tuple,
                    'side': side
                })
                
    return primer_candidates

def find_primer_flanked_overlaps(seq_record, primer_regions, overlap_regions, anchor_kmers, sequences_by_id, cfg):
    """Finds all valid primer pairs that flank overlaps within the specified regions."""
    forward_primers = find_primers_in_regions(seq_record, primer_regions, "forward", anchor_kmers, sequences_by_id, cfg)
    reverse_primers = find_primers_in_regions(seq_record, primer_regions, "reverse", anchor_kmers, sequences_by_id, cfg)
    
    primer_flanked_overlaps = []

    for reg in overlap_regions:
        reg_forward_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], forward_primers))
        reg_reverse_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], reverse_primers))
        if len(reg_forward_primers) > 0 and len(reg_reverse_primers) > 0:
            for fwd in reg_forward_primers:
                for rev in reg_reverse_primers:
                    overlap_start = fwd['pos'][0]
                    overlap_end = rev['pos'][1]
                    if overlap_end - overlap_start >= cfg['min_overlap'] and overlap_end - overlap_start <= cfg['max_overlap']:
                        primer_flanked_overlaps.append({
                            'forward': fwd,
                            'reverse': rev,
                            'pos': (overlap_start, overlap_end)
                        })
    return primer_flanked_overlaps

Running the abovedefined functions to get primer-flanked overlaps

In [ ]:
anchor_kmers = build_anchor_kmers(sequences, CONFIG['primer_3prime_anchor'])

primer_flanked_overlaps = {}

for seq in sequences:
    print(seq.id)
    primer_flanked_overlaps[seq.id] = find_primer_flanked_overlaps(seq, bg_regions[seq.id], bg_regions[seq.id], anchor_kmers, sequences_by_id, CONFIG)
